# dh2loop LLM-based Lithology Matching Demo

This notebook demonstrates how to use LLM-based lithology matching as an alternative to fuzzywuzzy in dh2loop.

## Prerequisites

1. Install dependencies: `pip install -r requirements_llm.txt`
2. Set up an LLM provider (Ollama recommended for local testing)
3. For Ollama: `ollama pull llama2` and `ollama serve`

In [ ]:
# Import required modules
import sys
sys.path.insert(0, '../dh2loop')

from dh2l_llm import create_matcher, load_litho_dictionary
import pandas as pd

## 1. Set Up the LLM Matcher

Configure your preferred LLM provider:

In [ ]:
# Option 1: Ollama (local, free)
config_ollama = {
    "provider": "ollama",
    "model": "llama2",
    "endpoint": "http://localhost:11434",
    "temperature": 0.0
}

# Option 2: AWS Bedrock (requires AWS credentials)
config_bedrock = {
    "provider": "bedrock",
    "model": "anthropic.claude-3-sonnet-20240229-v1:0",
    "temperature": 0.0
}

# Option 3: OpenRouter (requires API key)
config_openrouter = {
    "provider": "openrouter",
    "model": "openai/gpt-3.5-turbo",
    "api_key": "your-api-key-here",
    "temperature": 0.0
}

# Choose your provider
config = config_ollama  # Change this to use a different provider

# Create the matcher
matcher = create_matcher(config)
print(f"Initialized {config['provider']} matcher")

## 2. Load Lithology Dictionary

Load the standard lithology terms from the thesaurus:

In [ ]:
# For this demo, we'll use a sample dictionary
# In practice, load from: load_litho_dictionary('../thesauri/thesaurus_geology_lithology_code.csv')

litho_dict = [
    # Igneous rocks
    "granite", "basalt", "andesite", "rhyolite", "diorite", "gabbro",
    # Sedimentary rocks
    "sandstone", "limestone", "shale", "mudstone", "conglomerate", "breccia",
    "siltstone", "dolomite", "coal",
    # Metamorphic rocks
    "schist", "gneiss", "quartzite", "marble", "slate", "phyllite",
    # Minerals and ores
    "quartz", "calcite", "iron_ore", "gold_ore"
]

print(f"Loaded {len(litho_dict)} standard lithology terms")

## 3. Single Match Example

Match a single company lithology description:

In [ ]:
# Example company description
company_desc = "coarse grained granite with quartz veins"

# Match using LLM
best_match, score = matcher.match_lithology(company_desc, litho_dict, threshold=0.8)

print(f"Company description: {company_desc}")
print(f"Best match: {best_match}")
print(f"Confidence score: {score:.1f}")

## 4. Batch Processing Example

Process multiple descriptions at once:

In [ ]:
# Sample company lithology descriptions
company_data = [
    {"CompanyID": 1, "HoleID": "DH001", "Company_Litho": "coarse grained granite"},
    {"CompanyID": 2, "HoleID": "DH002", "Company_Litho": "volcanic basaltic rock"},
    {"CompanyID": 3, "HoleID": "DH003", "Company_Litho": "fine grained sandstone"},
    {"CompanyID": 4, "HoleID": "DH004", "Company_Litho": "metamorphic schist"},
    {"CompanyID": 5, "HoleID": "DH005", "Company_Litho": "carbonate limestone"},
    {"CompanyID": 6, "HoleID": "DH006", "Company_Litho": "iron rich ore"},
]

# Process batch
results = matcher.batch_match(company_data, litho_dict, threshold=0.8)

# Display results
df = pd.DataFrame(results)
print("\nMatching Results:")
print(df[['HoleID', 'Company_Litho', 'CET_Litho', 'Score']])

## 5. Comparison with Fuzzy Matching

Let's compare LLM matching with traditional fuzzy matching:

In [ ]:
from fuzzywuzzy import process, fuzz

# Test cases that show LLM advantages
test_cases = [
    "dark volcanic rock",
    "fine grained metamorphic",
    "coarse sedimentary sand",
    "carbonate rock layered"
]

print("Comparison: LLM vs Fuzzy Matching\n")
print(f"{'Description':<30} {'LLM Match':<15} {'Fuzzy Match':<15} {'LLM Score':<10} {'Fuzzy Score':<10}")
print("=" * 85)

for desc in test_cases:
    # LLM match
    llm_match, llm_score = matcher.match_lithology(desc, litho_dict)
    
    # Fuzzy match
    fuzzy_results = process.extractOne(desc, litho_dict, scorer=fuzz.token_set_ratio)
    fuzzy_match, fuzzy_score = fuzzy_results[0], fuzzy_results[1]
    
    print(f"{desc:<30} {llm_match:<15} {fuzzy_match:<15} {llm_score:<10.1f} {fuzzy_score:<10}")

## 6. Working with Real Data

Example of processing a CSV file from dh2loop:

In [ ]:
# Load your drill hole lithology data
# df = pd.read_csv('path/to/your/lithology_data.csv', encoding='ISO-8859-1')

# For this demo, we'll create sample data
df = pd.DataFrame([
    {"CollarID": 1, "Depth_From": 0, "Depth_To": 10, "Company_Litho": "weathered granite"},
    {"CollarID": 1, "Depth_From": 10, "Depth_To": 25, "Company_Litho": "fresh granite"},
    {"CollarID": 1, "Depth_From": 25, "Depth_To": 40, "Company_Litho": "basaltic dyke"},
])

print("Original data:")
print(df)

# Match lithologies
print("\nMatching lithologies...")
for idx, row in df.iterrows():
    match, score = matcher.match_lithology(row['Company_Litho'], litho_dict)
    df.at[idx, 'CET_Litho'] = match
    df.at[idx, 'Score'] = score

print("\nMatched data:")
print(df)

## 7. Advanced Configuration

Fine-tune the matching behavior:

In [ ]:
# Adjust threshold for classification
test_desc = "unknown rock material"

print("Effect of different thresholds:\n")
for threshold in [0.6, 0.7, 0.8, 0.9]:
    match, score = matcher.match_lithology(test_desc, litho_dict, threshold=threshold)
    print(f"Threshold {threshold}: {match} (score: {score:.1f})")

## 8. Export Results

Save matched lithology data to CSV:

In [ ]:
# Export to CSV
output_file = "matched_lithology_llm.csv"
df.to_csv(output_file, index=False, encoding='utf-8')
print(f"Results saved to {output_file}")

## Summary

This notebook demonstrated:
1. Setting up LLM-based matchers with different providers
2. Single and batch lithology matching
3. Comparison with fuzzy matching
4. Working with real drill hole data
5. Exporting results

### Next Steps
- Try different LLM providers (Bedrock, OpenRouter)
- Experiment with different models
- Integrate with existing dh2loop workflows
- Load your own lithology thesaurus
- Process larger datasets